by shizm

In [1]:
import os
from pathlib import Path
from ultralytics import YOLO
import json
import pandas as pd
from datetime import datetime

PROJECT_ROOT = Path("/home/shizm/DL_LABs/DL-lab2").resolve()

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RUNS_DIR = PROJECT_ROOT / "runs"
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
CONFIG_DIR = PROJECT_ROOT / "configs"
SRC_DIR = PROJECT_ROOT / "src"

DATA_YAML = CONFIG_DIR / "data.yaml"
SAMPLE_SUB = SUBMISSIONS_DIR / "sample_sub.csv"
TEST_IMAGES_DIR = DATA_DIR / "test_images" / "test_images"

os.chdir(PROJECT_ROOT)

RUNS_DIR.mkdir(exist_ok=True, parents=True)
SUBMISSIONS_DIR.mkdir(exist_ok=True, parents=True)

In [2]:
existing = [d.name for d in RUNS_DIR.glob("detect/train*")]
nums = [int(d.replace("train", "")) for d in existing if d.replace("train", "").isdigit()]
next_num = max(nums) + 1 if nums else 1

model = YOLO(MODELS_DIR / "yolo26s.pt")
model.train(
    data=str(DATA_YAML),
    epochs=60,
    imgsz=736,
    batch=8,
    project=str(RUNS_DIR / "detect"),
    name=f"train{next_num}",
    verbose=False,

    # Аугментация
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=5.0,
    perspective=0.001,
    flipud=0.1,
    fliplr=0.5,
    
    # Оптимизатор
    optimizer='SGD',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=5e-4
)

Ultralytics 8.4.21 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 7842MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/shizm/DL_LABs/DL-lab2/configs/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.8, hsv_v=0.4, imgsz=736, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/shizm/DL_LABs/DL-lab2/models/yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train1, nbs=64, nms=False, opset=None, o

/home/shizm/DL_LABs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1609.2±676.0 MB/s, size: 509.6 KB)
val: Scanning /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/labels.cache... 2000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2000/2000 72.3Mit/s 0.0s
train: /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/images/0209-10_00867800.jpg: 1 duplicate labels removed
train: /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/images/0209-10_00890700.jpg: 1 duplicate labels removed
train: /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/images/0209-10_00974200.jpg: 1 duplicate labels removed
train: /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/images/0209-27_00944300.jpg: 1 duplicate labels removed
train: /home

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dc72c038320>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [3]:
model_paths = list(RUNS_DIR.glob("detect/train*/weights/best.pt"))
best_model_path = max(model_paths, key=lambda p: p.stat().st_mtime)
best_model_name = best_model_path.parent.parent.name

In [4]:
model = YOLO(best_model_path)
metrics = model.val(
    data=str(DATA_YAML), 
    verbose=False,
    project=str(RUNS_DIR / "detect")
)

Ultralytics 8.4.21 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 7842MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7349.8±1104.9 MB/s, size: 444.0 KB)
val: Scanning /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/labels.cache... 2000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2000/2000 838.9Mit/s 0.0s
train: /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/images/0209-10_00867800.jpg: 1 duplicate labels removed
train: /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/images/0209-10_00890700.jpg: 1 duplicate labels removed
train: /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/images/0209-10_00974200.jpg: 1 duplicate labels removed
train: /home/shizm/DL_LABs/DL-lab2/data/yolo_dataset/yolo_dataset/train/images/0209-27_00944300.jpg: 1 duplicate labels removed
train: /home/shizm/DL_LABs/DL-lab

In [5]:
pred_name = f"predict_{best_model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

results = model.predict(
    source=str(TEST_IMAGES_DIR),
    save_txt=True,
    save_conf=True,
    stream=True,
    project=str(RUNS_DIR / "detect"),
    name=pred_name,
    verbose=False
)

for _ in results:
    pass

pred_labels_dir = RUNS_DIR / "detect" / pred_name / "labels"

Results saved to /home/shizm/DL_LABs/DL-lab2/runs/detect/predict_train1_20260312_005758
4306 labels saved to /home/shizm/DL_LABs/DL-lab2/runs/detect/predict_train1_20260312_005758/labels


In [6]:
def build_submission(solution_csv, preds_dir, output_csv, keep_only_class=1):
    sol = pd.read_csv(Path(solution_csv))
    pdir = Path(preds_dir)
    rows = []
    
    for idx, image_name in enumerate(sol["image_name"].astype(str)):
        pred_file = pdir / f"{Path(image_name).stem}.txt"
        boxes = []
        
        if pred_file.exists():
            content = pred_file.read_text().strip()
            if content:
                for ln in content.splitlines():
                    parts = ln.split()
                    if len(parts) >= 6:
                        try:
                            cls = int(float(parts[0]))
                            if cls == keep_only_class:
                                boxes.append([float(parts[1]), float(parts[2]), 
                                            float(parts[3]), float(parts[4]), 
                                            float(parts[5])])
                        except:
                            pass
        
        rows.append({"id": idx, "image_name": image_name, 
                    "boxes": json.dumps(boxes, separators=(",", ":"))})
    
    pd.DataFrame(rows).to_csv(output_csv, index=False)

submission_path = SUBMISSIONS_DIR / f"submission_{best_model_name}.csv"
build_submission(str(SAMPLE_SUB), str(pred_labels_dir), str(submission_path))